# 7. Oxygen-limited growth

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sisyga/biolgca/blob/aidevelop/docs/source/tutorials/07_oxygen_limited_growth.ipynb)

Cells and the molecules around them act on each other. A colony of tumour
cells takes up the oxygen that diffuses in from the surrounding tissue;
where too little arrives, cells stop dividing and die. The colony then grows
as a rim of dividing cells around a hypoxic core, as tumour spheroids do.

BioLGCA describes such molecules as **fields**: one value per node, updated
by an operator of the pipeline that solves a reaction–diffusion equation, in
its listed place among the cell operators.

**Learning objectives**

- declare a field and the `pde` operator that updates it;
- convert physical parameters to lattice units and choose a solver;
- let division and death respond to the field;
- record fields and plot them with the cells; and
- let a trait evolve that sets both division and consumption.

In [ ]:
# In Google Colab this cell installs BioLGCA (about a minute); elsewhere it does nothing.
import importlib.util
import subprocess
import sys

if "google.colab" in sys.modules and importlib.util.find_spec("lgca") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "biolgca @ git+https://github.com/sisyga/biolgca@aidevelop"], check=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from lgca import LatticeState
from lgca.fields import PDESpec
from lgca.model import (
    AnalysisSpec,
    Description,
    ModelSpec,
    SpaceSpec,
    StateSpec,
    TimeSpec,
    build_model,
    run_model,
)
from lgca.pipeline import InteractionPipelineSpec
from lgca.simulation import DensityRecorder, FieldRecorder, PopulationRecorder, Schedule

## Oxygen in lattice units

The oxygen concentration $c$ obeys

$$\partial_t c = D\,\Delta c - L(c)\,c,$$

with diffusion coefficient $D$ and a loss rate $L$ from uptake by the cells.
Uptake saturates: a node with $n$ cells takes up $\mu\, n\, c / (K_m + c)$
per time step, Michaelis–Menten kinetics with maximal rate $\mu$ per cell.

The `pde` operator works in lattice units: lengths in nodes, times in LGCA
steps. A diffusion coefficient $D$ in µm²/s becomes $D\,\tau/\varepsilon^2$
for nodes of size $\varepsilon$ and steps of duration $\tau$. For oxygen in
tissue, cells of 20 µm and one step per hour:

In [ ]:
D_oxygen = 2e3  # µm² / s
node_size = 20.0  # µm
step = 3600.0  # s

D_lattice = D_oxygen * step / node_size**2
print(f"D = {D_lattice:.0f} nodes² per step")

Within one step oxygen spreads over $\sqrt{D} \approx 130$ nodes, far more
than a cell moves. Oxygen therefore relaxes to equilibrium with the current
cells long before they change: the **quasi-steady** assumption, which the
solver `"steady"` makes. At every step it solves
$D\,\Delta c - L(c)\,c = 0$, with the concentration 1 beyond the edge of the
lattice, where the surrounding tissue supplies oxygen.

A steady field depends only on the ratio of $D$ to the uptake, through the
distance over which oxygen penetrates a colony,
$\lambda = \sqrt{D / (\mu n)}$ for $c \gg K_m$. With $\mu = 20$ per cell and
step, 8 cells per node give $\lambda \approx 11$ nodes.

## The model

One time step is:

1. `pde`: oxygen at equilibrium with the cells;
2. `birth_death`: a cell divides with a probability that needs oxygen, and
   dies with one that rises where oxygen runs out. Both are switching
   probabilities in the Hill form (see *Switching that responds to the
   surroundings* in the concepts): `{"max": 0.1, "hill": [{"name": "field",
   "field": "oxygen", "K": 0.3, "n": 2}]}` is $0.1\,c^2 / (0.3^2 + c^2)$, and
   a negative `n` turns the response around;
3. `go_or_rest`: a cell rests with probability 0.8 and moves otherwise, so
   the colony stays compact;
4. `random_walk` over the velocity channels, then propagation.

The model has no volume exclusion; a node holds about `capacity = 8` cells
before crowding stops division.

In [ ]:
size = 100


def spheroid_spec(uptake=20.0, steps=200):
    return ModelSpec(
        description=Description(title="Oxygen-limited growth"),
        space=SpaceSpec(geometry="square", dims=(size, size), boundary="reflecting"),
        state=StateSpec(
            restchannels=1,
            volume_exclusion=False,
            capacity=8,
            initializer={"name": "region", "parameters": {"extent": [12, 12], "density": 0.8}},
            fields={"oxygen": 1.0},
        ),
        time=TimeSpec(steps=steps, seed=3),
        dynamics=InteractionPipelineSpec(
            operators=[
                PDESpec(
                    field="oxygen",
                    diffusion=2e4,
                    cells=[{"uptake": uptake, "saturation": 0.05}],
                    boundary={"value": 1.0},
                    solver="steady",
                ),
                {"name": "birth_death", "parameters": {
                    "birth_rate": {"max": 0.1, "hill": [
                        {"name": "field", "field": "oxygen", "K": 0.3, "n": 2}]},
                    "death_rate": {"max": 0.1, "hill": [
                        {"name": "field", "field": "oxygen", "K": 0.05, "n": -4}]},
                }},
                {"name": "go_or_rest", "parameters": {"probability": 0.8}},
                {"name": "random_walk", "parameters": {"channels": "velocity"}},
            ],
        ),
        analysis=AnalysisSpec(
            observers=[
                PopulationRecorder(),
                DensityRecorder(schedule=Schedule(every=50)),
                FieldRecorder(["oxygen"], schedule=Schedule(every=50)),
            ],
        ),
    )


spheroid = run_model(spheroid_spec(), showprogress=False)
print(spheroid.metadata["fields"]["oxygen"])

The metadata of the run show how the field was solved: the `"amg"`
backend (conjugate gradients with an algebraic multigrid preconditioner),
at most a few linear iterations per solve, and a few repetitions per step
for the saturating uptake, which depends on the concentration.

`FieldRecorder` recorded the oxygen every 50 steps, `DensityRecorder` the
cells at the same steps:

In [ ]:
steps = spheroid.data.steps("oxygen")
fig, axes = plt.subplots(2, 4, figsize=(11, 5.6), constrained_layout=True)
for column, index in enumerate(range(1, len(steps))):
    cells = axes[0, column].imshow(spheroid.data["density"][index].T, origin="lower", vmin=0, vmax=10)
    oxygen = axes[1, column].imshow(spheroid.data["oxygen"][index].T, origin="lower", vmin=0, vmax=1)
    axes[0, column].set_title(f"step {steps[index]}")
for axis in axes.flat:
    axis.set(xticks=[], yticks=[])
fig.colorbar(cells, ax=axes[0], label="cells per node", shrink=0.8)
fig.colorbar(oxygen, ax=axes[1], label="oxygen", shrink=0.8)
plt.show()
plt.close(fig)

The growing colony draws the oxygen down. Once its radius exceeds the
penetration length, the centre becomes hypoxic, cells there die, and the
colony continues as a ring.

## A proliferating rim around a hypoxic core

Averaged over rings around the centre, the final state shows where cells
divide and die. The probabilities follow from the local oxygen by the two
Hill functions of the model:

In [ ]:
x, y = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
radius = np.hypot(x - (size - 1) / 2, y - (size - 1) / 2).astype(int).ravel()


def ring_mean(values):
    """The mean over nodes at the same distance from the centre."""
    return np.bincount(radius, weights=values.ravel()) / np.bincount(radius)


oxygen = spheroid.data["oxygen"][-1]
density = spheroid.data["density"][-1]
birth = 0.1 * oxygen**2 / (0.3**2 + oxygen**2)
death = 0.1 * 0.05**4 / (0.05**4 + oxygen**4)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
axes[0].plot(ring_mean(density), color="black", label="cells per node")
axes[0].set(xlabel="distance from the centre (nodes)", ylabel="cells per node", xlim=(0, 50))
twin = axes[0].twinx()
twin.plot(ring_mean(oxygen), color="tab:blue", label="oxygen")
twin.set(ylabel="oxygen", ylim=(0, 1.05))
axes[1].plot(ring_mean(birth * density), label="divisions per step")
axes[1].plot(ring_mean(death * density), label="deaths per step")
axes[1].set(xlabel="distance from the centre (nodes)", ylabel="per node", xlim=(0, 50))
axes[1].legend()
plt.show()
plt.close(fig)

Divisions happen in a band at the edge of the colony, where cells and
oxygen meet; deaths inside it, where oxygen has fallen below the death
threshold `K = 0.05`. The thickness of the proliferating rim is set by the
penetration length.

## Growth limited by the rim

Without uptake, oxygen stays at 1 everywhere, and cells divide wherever
their node has room. Growth slows in both colonies, since crowded nodes in
the interior leave room only at the edge; with uptake the interior also
starves, and the colony grows more slowly still:

In [ ]:
control = run_model(spheroid_spec(uptake=0.0), showprogress=False)

fig, axis = plt.subplots(figsize=(6, 3.6), constrained_layout=True)
steps = spheroid.data.steps("population")
axis.plot(steps, control.data["population"], label="no uptake")
axis.plot(steps, spheroid.data["population"], label="uptake 20 per cell")
axis.set(xlabel="time step (h)", ylabel="cells")
axis.legend()
plt.show()
plt.close(fig)

## Consumption as an evolving trait

Cells differ, and their differences are inherited. In an identity-based
model every cell carries its own maximal division rate `r_b`, which
daughters inherit with small random changes (see tutorial 5). Suppose that
faster division costs oxygen: a cell consumes in proportion to its `r_b`.
Then a fast-dividing cell gains for itself, but the oxygen it takes is
missing for its neighbours.

Three changes turn the model into this one:

- `identity_based=True` and the trait `r_b`; `"max": "r_b"` makes each
  cell's maximal division rate its own trait, while division still needs
  oxygen;
- a `mutation` of `r_b` in daughters;
- uptake `"r_b"`: each cell consumes at the rate of its trait. The steady
  field depends only on $D$ over the uptake, so $D = 100$ with uptake
  `r_b = 0.1` is the same as $D = 2 \cdot 10^4$ with uptake 20 above.

For comparison, a colony in which every cell consumes 0.1, whatever its
division rate.

In [ ]:
def evolving_spec(consumption, steps=300):
    return ModelSpec(
        description=Description(title="Evolving division and consumption"),
        space=SpaceSpec(geometry="square", dims=(size, size), boundary="reflecting"),
        state=StateSpec(
            restchannels=1,
            volume_exclusion=False,
            capacity=8,
            identity_based=True,
            traits={"r_b": 0.1},
            initializer={"name": "region", "parameters": {"extent": [12, 12], "density": 0.8}},
            fields={"oxygen": 1.0},
        ),
        time=TimeSpec(steps=steps, seed=5),
        dynamics=InteractionPipelineSpec(
            operators=[
                PDESpec(
                    field="oxygen",
                    diffusion=100.0,
                    cells=[{"uptake": consumption, "saturation": 0.05}],
                    boundary={"value": 1.0},
                    solver="steady",
                ),
                {"name": "birth_death", "parameters": {
                    "birth_rate": {"max": "r_b", "hill": [
                        {"name": "field", "field": "oxygen", "K": 0.3, "n": 2}]},
                    "death_rate": {"max": 0.1, "hill": [
                        {"name": "field", "field": "oxygen", "K": 0.05, "n": -4}]},
                    "mutation": {"probability": 0.2, "traits": {"r_b": {
                        "distribution": "normal", "scale": 0.02, "bounds": [0, 1]}}},
                }},
                {"name": "go_or_rest", "parameters": {"probability": 0.8}},
                {"name": "random_walk", "parameters": {"channels": "velocity"}},
            ],
        ),
    )

To follow the mean trait over time, we build the models and step them
ourselves. `LatticeState(model.lgca).cells` is the table of living cells,
with one entry per cell for every trait:

In [ ]:
histories = {}
evolved = {}
for label, consumption in [("consumption r_b", "r_b"), ("consumption 0.1", 0.1)]:
    model = build_model(evolving_spec(consumption))
    history = []
    for step in range(301):
        if step % 10 == 0:
            r_b = LatticeState(model.lgca).cells["r_b"]
            history.append((step, len(r_b), r_b.mean()))
        if step < 300:
            model.step()
    histories[label] = np.array(history)
    evolved[label] = model

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
for label, history in histories.items():
    axes[0].plot(history[:, 0], history[:, 2], label=label)
    axes[1].plot(history[:, 0], history[:, 1], label=label)
axes[0].set(xlabel="time step (h)", ylabel="mean r_b")
axes[1].set(xlabel="time step (h)", ylabel="cells")
axes[0].legend()
plt.show()
plt.close(fig)

Faster division evolves in both colonies: the cells that divide are those
at the rim, where oxygen is plentiful, and among them the fast ones leave
more daughters. The cost of consumption hardly holds this back, because it
falls on everyone nearby, not on the consumer alone. It shows in the colony
as a whole, which ends up smaller when consumption rises with `r_b`. One
run can mislead, so we checked six seeds (with `lgca.study.sweep`, see
tutorial 3): the mean `r_b` after 300 steps varied between runs about as much as
between the two models (0.13 to 0.16 in both), while the colony with costly
consumption was smaller in every run (8 500 to 9 700 cells, against 11 300 to
12 300).

Where in the colony are the fast dividers? The mean trait per node of the
colony whose consumption evolves:

In [ ]:
model = evolved["consumption r_b"]
state = LatticeState(model.lgca)
table = state.cells
counts = np.bincount(table.index, minlength=size * size)
total = np.bincount(table.index, weights=table["r_b"], minlength=size * size)
mean_r_b = np.where(counts > 0, total / np.maximum(counts, 1), np.nan).reshape(size, size)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), constrained_layout=True)
image = axes[0].imshow(mean_r_b.T, origin="lower", cmap="viridis")
fig.colorbar(image, ax=axes[0], label="mean r_b")
field = axes[1].imshow(model.lgca.oxygen[model.lgca.nonborder].T, origin="lower", vmin=0, vmax=1)
fig.colorbar(field, ax=axes[1], label="oxygen")
for axis, title in zip(axes, ("division rate", "oxygen")):
    axis.set(title=title, xticks=[], yticks=[])
plt.show()
plt.close(fig)

Fast dividers are found at the outer edge, often in sectors: a lineage that
arises at the expanding front rides the expansion outward, while the cells
behind it no longer divide.

## Exercises

1. Change the uptake to 10 and to 40 per cell. How does the thickness of
   the proliferating rim change? Compare it with
   $\lambda = \sqrt{D / (\mu n)}$.
2. Supply oxygen from one side only:
   `boundary={"x-": {"value": 1.0}, "default": "no_flux"}`. Where does the
   colony grow?
3. Let hypoxic cells migrate instead of resting: give `go_or_rest` a
   probability that rises with oxygen, e.g.
   `{"max": 0.9, "hill": [{"name": "field", "field": "oxygen", "K": 0.2}]}`.
   How does the shape of the colony change?
4. Replace `solver="steady"` by `"implicit"`. With `D = 2e4` the result
   hardly changes. With `D = 20` oxygen is slow: what happens to the core?
5. In the evolution model, set the mutation `scale` to 0. What remains of
   the difference between the two colonies?